### Pipeline 1- doc load, split doc, embeding,store vector db

In [40]:
from dotenv import load_dotenv
load_dotenv()

True

In [53]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate

In [42]:
loader = PyPDFLoader("../data/medical_report.pdf")
docs = loader.load()
len(docs)

9

In [43]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
splitted_data = splitter.split_documents(docs)
len(splitted_data)

26

In [44]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [45]:
vector_store = Chroma.from_documents(
    documents=splitted_data,
    embedding=embeddings
)

In [47]:
query = "Machine Learnings and Data Science Content"
data = vector_store.similarity_search(query=query)
len(data)

4

In [48]:
context = "" 
for doc in data:
    context += doc.page_content + "\n"

In [49]:
llm = ChatOpenAI(model="gpt-4o-mini")

In [50]:
res =  llm.invoke(f"""can you provide me the answer based on provided content for my question ,content:{context} and question: {query} """)

In [51]:
print(res.content)

Based on the content provided, here's a summary related to Machine Learning and Data Science:

### Machine Learning and Data Science Content Overview

#### Module 6: Machine Learning – I (Supervised Learning)
- **Duration:** 4 weeks in Month 6
- **Topics Covered:**
  - Understanding the ML pipeline.
  - Regression techniques: Linear and Logistic regression.
  - Algorithms: Decision Tree, Random Forest, and K-Nearest Neighbors (KNN).
  - Concepts of train-test split and model evaluation.
- **Tools Used:**
  - Scikit-learn, Google Colab, ChatGPT, PyCaret (optional).
- **Mini Projects:**
  - Loan Approval or House Price Prediction.
  - Predicting Diabetes from a health dataset.

#### Module 7: Machine Learning – II (Unsupervised Learning & Feature Engineering)
- **Duration:** 3 weeks in Month 7
- **Topics Covered:**
  - KMeans Clustering.
  - Dimensionality Reduction using PCA (Principal Component Analysis).
  - Feature selection, encoding, and scaling techniques.
  - Model tuning with Gr

### Context generation | prompt | llm | strparser 

In [55]:
def get_context(query:str):
    data = vector_store.similarity_search(query=query)
    context = "" 
    for doc in data:
        context += doc.page_content + "\n"

    return {
        "context":context,
        "question":query
    }
    

In [56]:
prompt = PromptTemplate.from_template("""
    you are a helpful assistant provide answer based on the context for user question,if you dont know the answer, then you can say then i dont know.context:{context},question:{question}
""")

In [57]:
rag_chain = get_context | prompt | llm

In [62]:
result = rag_chain.invoke("what is the content of java dsa course?")

In [63]:
print(result.content)

I don't know.
